In [1]:
# CELL 1: Imports and Setup
import os, glob, numpy as np, pandas as pd
import librosa, torch, torch.nn as nn, timm
import warnings
warnings.filterwarnings('ignore')

BASE_PATH    = '/kaggle/input/competitions/birdclef-2026'
SR           = 32000
DURATION     = 5
N_MELS       = 128
NUM_CLASSES  = 206

sample_sub   = pd.read_csv(f'{BASE_PATH}/sample_submission.csv')
species_cols = [c for c in sample_sub.columns if c != 'row_id']
train_df     = pd.read_csv(f'{BASE_PATH}/train.csv')
species_list = sorted(train_df['primary_label'].unique())

print("Setup done!")
print("Species:", len(species_cols))

Setup done!
Species: 234


In [2]:
# CELL 2: Load Model
class BirdModel(nn.Module):
    def __init__(self, num_classes=206):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b4',
            pretrained=False,
            in_chans=1,
            num_classes=num_classes
        )
    def forward(self, x):
        return self.backbone(x)

model_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file == 'best_model_v5.pth':
            model_path = os.path.join(root, file)

print(f"Model found: {model_path}")
model = BirdModel(num_classes=NUM_CLASSES)
model.load_state_dict(
    torch.load(model_path, map_location='cpu'))
model.eval()
print("Model loaded successfully!")

Model found: /kaggle/input/datasets/pythonophile/birdclef-2026-model-v2/best_model_v5.pth
Model loaded successfully!


In [3]:
def audio_to_mel(y):
    mel    = librosa.feature.melspectrogram(
        y=y, sr=SR, n_mels=N_MELS, fmin=20, fmax=16000)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mean   = mel_db.mean()
    std    = mel_db.std() + 1e-8
    mel_db = (mel_db - mean) / std
    return mel_db

test_files = sorted(
    glob.glob(f'{BASE_PATH}/test_soundscapes/*.ogg'))
print(f"Test files found: {len(test_files)}")

all_row_ids = []
all_preds   = []

if len(test_files) == 0:
    print("⚠️ No test files - using sample submission")
    final_sub = sample_sub.copy()
else:
    for i, fpath in enumerate(test_files):
        fname = os.path.basename(fpath).replace('.ogg','')
        y, _  = librosa.load(fpath, sr=SR)
        total = int(len(y) / SR)

        for start in range(0, total, DURATION):
            chunk = y[start*SR:(start+DURATION)*SR]
            if len(chunk) < SR:
                continue
            if len(chunk) < SR * DURATION:
                chunk = np.pad(
                    chunk,(0, SR*DURATION - len(chunk)))

            mel = audio_to_mel(chunk)
            t   = torch.tensor(
                mel, dtype=torch.float32
            ).unsqueeze(0).unsqueeze(0)

            with torch.no_grad():
                out   = model(t)
                probs = torch.sigmoid(
                    out).cpu().numpy()[0]

            all_row_ids.append(f'{fname}_{start+DURATION}')
            all_preds.append(probs)

        if (i+1) % 5 == 0:
            print(f"Processed {i+1}/{len(test_files)}")

    final_sub           = pd.DataFrame(
        index=range(len(all_row_ids)),
        columns=['row_id'] + species_cols)
    final_sub['row_id'] = all_row_ids
    final_sub[species_cols] = 0.004274

    for i, preds in enumerate(all_preds):
        for j, sp in enumerate(species_list):
            if str(sp) in species_cols:
                final_sub.at[i, str(sp)] = float(preds[j])

print("Inference done!")

Test files found: 0
⚠️ No test files - using sample submission
Inference done!


In [4]:
# CELL 4: Save Submission
final_sub.to_csv('/kaggle/working/submission.csv', index=False)
print("Submission saved!")
print("Shape:", final_sub.shape)

Submission saved!
Shape: (3, 235)
